In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

In [6]:
meta = pd.read_csv('../data/spatial/metadata/2d_data_tracking_visium.csv', sep=',', index_col='Section_ID_clean')
meta = meta[~pd.isnull(meta.index)]
meta

,Case,Section ID,Cancer type,Tissue site,Primary/met,spaceranger_outs,alla_annotation_fp,cloupe_filepath,annotation_fp,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
Section_ID_clean,,,,,,,,,,,,,,,,
HT206B1-S1Fc1U1Z1B1,HT206B1,HT206B1-U1_ST_Bn1,BRCA,Breast,Primary,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HT206B1-S1Fc1U2Z1B1,HT206B1,HT206B1-S1Fc1U2Z1B1,BRCA,Breast,Primary,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HT206B1-S1Fc1U3Z1B1,HT206B1,HT206B1-S1Fc1U3Z1B1,BRCA,Breast,Primary,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HT206B1-S1Fc1U4Z1B1,HT206B1,HT206B1-S1Fc1U4Z1B1,BRCA,Breast,Primary,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HT206B1-S1Fc1U5Z1B1,HT206B1,HT206B1-S1Fc1U5Z1B1,BRCA,Breast,Primary,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HT297B1-S1H1Fc2U1Z1Bs1,HT297B1,HT297B1-S1H1Fc2U1Z1Bs1,BRCA,Breast,Primary,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HT308B1-S1H1Fc2U1Z1Bs1,HT308B1,HT308B1-S1H1Fc2U1Z1Bs1,BRCA,Breast,Primary,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HT308B1-S1H5Fc2U1Z1Bs1,HT308B1,HT308B1-S1H5Fc2U1Z1Bs1,BRCA,Breast,Primary,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HT308B1-S1H4Fc2U1Z1Bs1,HT308B1,HT308B1-S1H4Fc2U1Z1Bs1,BRCA,Breast,Primary,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,/diskmnt/Projects/Users/estorrs/mushroom/data/...,/diskmnt/Datasets/Spatial_Transcriptomics/outp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
sid_mapping = {
    'BRCA_206B1': 'HT206B1-S1Fc1U1Z1B1',
    'BR_206B1_U2': 'HT206B1-S1Fc1U2Z1B1',
    'BR_206B1_U3': 'HT206B1-S1Fc1U3Z1B1',
    'BR_206B1_U4': 'HT206B1-S1Fc1U4Z1B1',
    'BR_206B1_U5': 'HT206B1-S1Fc1U5Z1B1',
    'HT297B1H1U1': 'HT297B1-S1H1Fc2U1Z1Bs1',
    'HT308B1-S1H1': 'HT308B1-S1H1Fc2U1Z1Bs1',
    'HT308B1-S1H4': 'HT308B1-S1H4Fc2U1Z1Bs1',
    'HT308B1-S1H5': 'HT308B1-S1H5Fc2U1Z1Bs1',
    'HT323B1-S1H1': 'HT323B1-S1H1Fc2U1Z1Bs1',
    'HT339B1-S1H3-U1': 'HT339B1-S1H3Fc2U1Z1Bs1',
    'HT339B1-S1H3-U2': 'HT339B1-S1H3Fc2U2Bs2',
    'HT397B1-S1H2Fs4U1Bp1': 'HT397B1-S1H2Fs4U1Bp1',
    'HT397B1-S1H3Fs1U1Bp1': 'HT397B1-S1H3Fs1U1Bp1',
    'HT480B1-S1H1Fs1U1Bp1': 'HT480B1-S1H1Fs1U1Bp1',
    'HT480B1-S1H3Fs1U1Bp1': 'HT480B1-S1H3Fs1U1Bp1',
    'HT486B1-S1H1Fs1U1Bp1': 'HT486B1-S1H1Fs1U1Bp1'
}

In [32]:
def get_label(x):
    if pd.isnull(x):
        return np.nan
    if 'DCIS ' in x:
        return 'DCIS'
    if 'IDC ' in x:
        return 'IDC'
    if 'Normal ' in x:
        return 'Normal'
    return np.nan


annots = pd.read_csv('../data/spatial/annotations/visium/All_ST_annotation_09202022.txt', sep='\t')
annots.index = [f'{sid_mapping[sid]}_{barcode}' for barcode, sid in annots[['Barcode', 'Sample_ID']].values]
annots['morphology'] = [get_label(x) for x in annots['celltypespecific']]
annots = annots[~pd.isnull(annots['morphology'])]
annots['sid'] = [sid_mapping[x] for x in annots['Sample_ID']]
annots['label'] = annots['celltypespecific']
annots

,Barcode,celltypespecific,Sample_ID,Preservation,morphology,sid,label
HT308B1-S1H4Fc2U1Z1Bs1_AAACCTAAGCAGCCGG-1,AAACCTAAGCAGCCGG-1,IDC 4,HT308B1-S1H4,OCT,IDC,HT308B1-S1H4Fc2U1Z1Bs1,IDC 4
HT308B1-S1H4Fc2U1Z1Bs1_AAACGAAGAACATACC-1,AAACGAAGAACATACC-1,DCIS 15,HT308B1-S1H4,OCT,DCIS,HT308B1-S1H4Fc2U1Z1Bs1,DCIS 15
HT308B1-S1H4Fc2U1Z1Bs1_AAATGCTCGTTACGTT-1,AAATGCTCGTTACGTT-1,IDC 7,HT308B1-S1H4,OCT,IDC,HT308B1-S1H4Fc2U1Z1Bs1,IDC 7
HT308B1-S1H4Fc2U1Z1Bs1_AAATTAACGGGTAGCT-1,AAATTAACGGGTAGCT-1,IDC 8,HT308B1-S1H4,OCT,IDC,HT308B1-S1H4Fc2U1Z1Bs1,IDC 8
HT308B1-S1H4Fc2U1Z1Bs1_AAATTGCGGCGGTTCT-1,AAATTGCGGCGGTTCT-1,IDC 7,HT308B1-S1H4,OCT,IDC,HT308B1-S1H4Fc2U1Z1Bs1,IDC 7
...,...,...,...,...,...,...,...
HT297B1-S1H1Fc2U1Z1Bs1_TTCGGGCGCTAGTCTT-1,TTCGGGCGCTAGTCTT-1,DCIS 8,HT297B1H1U1,OCT,DCIS,HT297B1-S1H1Fc2U1Z1Bs1,DCIS 8
HT297B1-S1H1Fc2U1Z1Bs1_TTCGTTCAACGAAGTT-1,TTCGTTCAACGAAGTT-1,IDC 5,HT297B1H1U1,OCT,IDC,HT297B1-S1H1Fc2U1Z1Bs1,IDC 5
HT297B1-S1H1Fc2U1Z1Bs1_TTCTGTTTCCTGTCGC-1,TTCTGTTTCCTGTCGC-1,IDC 1,HT297B1H1U1,OCT,IDC,HT297B1-S1H1Fc2U1Z1Bs1,IDC 1
HT297B1-S1H1Fc2U1Z1Bs1_TTCTTCCCTTTGATAT-1,TTCTTCCCTTTGATAT-1,DCIS 7,HT297B1H1U1,OCT,DCIS,HT297B1-S1H1Fc2U1Z1Bs1,DCIS 7


In [23]:
directory = Path('../data/spatial/annotations/visium/')
fps = [directory / path for path in os.listdir(directory) if 'All' not in path]
fps

[PosixPath('../data/spatial/annotations/visium/HT480B1-S1H1Fs1U2Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/SP060Z1-Fp1U1Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/HT397B1-S1H3A1U61Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/HT397B1-S1H3A1U41Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/SP017Z1-Fp1B27Us1_1Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/SP008Z1-Fp1F15U1Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/HT425B1-S1H2Fs1U2Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/HT486B1-S1H1Fs1U2Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/HT397B1-S1H3A1U21Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/SP017Z1-Fp1B27Us1_16Bp1.csv'),
 PosixPath('../data/spatial/annotations/visium/HT397B1-S1H3A1U1Bp1.csv')]

In [31]:
dfs = []
for fp in fps:
    sid = str(fp).split('/')[-1].split('.')[0]
    df = pd.read_csv(fps[0], sep=',')
    df.index = [f'{sid}_{x}' for x in df['Barcode']]
    df['sid'] = sid
    dfs.append(df)
df = pd.concat(dfs)
df
    

,Barcode,label,sid
HT480B1-S1H1Fs1U2Bp1_AAATCGTGTACCACAA-1,AAATCGTGTACCACAA-1,IDC 1,HT480B1-S1H1Fs1U2Bp1
HT480B1-S1H1Fs1U2Bp1_AAATGGTCAATGTGCC-1,AAATGGTCAATGTGCC-1,IDC 1,HT480B1-S1H1Fs1U2Bp1
HT480B1-S1H1Fs1U2Bp1_AACAACTGGTAGTTGC-1,AACAACTGGTAGTTGC-1,IDC 1,HT480B1-S1H1Fs1U2Bp1
HT480B1-S1H1Fs1U2Bp1_AACGTACTGTGGGTAC-1,AACGTACTGTGGGTAC-1,IDC 1,HT480B1-S1H1Fs1U2Bp1
HT480B1-S1H1Fs1U2Bp1_AACGTCAGACTAGTGG-1,AACGTCAGACTAGTGG-1,IDC 1,HT480B1-S1H1Fs1U2Bp1
...,...,...,...
HT397B1-S1H3A1U1Bp1_TTGGATTGGGTACCAC-1,TTGGATTGGGTACCAC-1,IDC 2,HT397B1-S1H3A1U1Bp1
HT397B1-S1H3A1U1Bp1_TTGTAATCCGTACTCG-1,TTGTAATCCGTACTCG-1,IDC 1,HT397B1-S1H3A1U1Bp1
HT397B1-S1H3A1U1Bp1_TTGTATCACACAGAAT-1,TTGTATCACACAGAAT-1,DCIS 3,HT397B1-S1H3A1U1Bp1
HT397B1-S1H3A1U1Bp1_TTGTCGTTCAGTTACC-1,TTGTCGTTCAGTTACC-1,IDC 2,HT397B1-S1H3A1U1Bp1


In [34]:
annots = pd.concat((annots[['sid', 'Barcode', 'label']], df[['sid', 'Barcode', 'label']]))
annots

,sid,Barcode,label
HT308B1-S1H4Fc2U1Z1Bs1_AAACCTAAGCAGCCGG-1,HT308B1-S1H4Fc2U1Z1Bs1,AAACCTAAGCAGCCGG-1,IDC 4
HT308B1-S1H4Fc2U1Z1Bs1_AAACGAAGAACATACC-1,HT308B1-S1H4Fc2U1Z1Bs1,AAACGAAGAACATACC-1,DCIS 15
HT308B1-S1H4Fc2U1Z1Bs1_AAATGCTCGTTACGTT-1,HT308B1-S1H4Fc2U1Z1Bs1,AAATGCTCGTTACGTT-1,IDC 7
HT308B1-S1H4Fc2U1Z1Bs1_AAATTAACGGGTAGCT-1,HT308B1-S1H4Fc2U1Z1Bs1,AAATTAACGGGTAGCT-1,IDC 8
HT308B1-S1H4Fc2U1Z1Bs1_AAATTGCGGCGGTTCT-1,HT308B1-S1H4Fc2U1Z1Bs1,AAATTGCGGCGGTTCT-1,IDC 7
...,...,...,...
HT397B1-S1H3A1U1Bp1_TTGGATTGGGTACCAC-1,HT397B1-S1H3A1U1Bp1,TTGGATTGGGTACCAC-1,IDC 2
HT397B1-S1H3A1U1Bp1_TTGTAATCCGTACTCG-1,HT397B1-S1H3A1U1Bp1,TTGTAATCCGTACTCG-1,IDC 1
HT397B1-S1H3A1U1Bp1_TTGTATCACACAGAAT-1,HT397B1-S1H3A1U1Bp1,TTGTATCACACAGAAT-1,DCIS 3
HT397B1-S1H3A1U1Bp1_TTGTCGTTCAGTTACC-1,HT397B1-S1H3A1U1Bp1,TTGTCGTTCAGTTACC-1,IDC 2


In [ ]:
annots.to_csv('../data/spatial/annotations/visium/all_annotations.txt', sep='')